In [ ]:
# 1. Install necessary Python libraries
# Using --quiet to keep the installation output clean
!pip install --quiet sentence-transformers pandas yt-dlp faiss-cpu
!pip install --quiet "transformers>=4.38.1" torch accelerate datasets[audio] ipywidgets

# 2. IMPORTANT: User must have ffmpeg installed if running locally, though Colab often has it.

import torch
from transformers import pipeline
import faiss
from sentence_transformers import SentenceTransformer
import pandas as pd
import os
import re
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import subprocess
import threading
import time
import gc # Import garbage collector

# --- Configuration and Core Logic ---

SBERT_MODEL_NAME = "all-MiniLM-L6-v2"
AUDIO_FILE = "video_audio.mp3"
TEMP_VIDEO_FILE = "uploaded_video.tmp" # For uploaded files

# --- HIGH-ACCURACY & HIGH-PERFORMANCE MODEL LOADING ---
# All models are pre-loaded for maximum speed, assuming high memory.
print("Loading high-accuracy models... Please wait.")
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype_torch = torch.float16 if torch.cuda.is_available() else torch.float32

# Using the most accurate 'large' distilled model
SPEECH_PIPE = pipeline(
    "automatic-speech-recognition",
    model="distil-whisper/distil-large-v2",
    torch_dtype=dtype_torch,
    device=DEVICE
)

# Pre-load the search model
SBERT_MODEL = SentenceTransformer(SBERT_MODEL_NAME, device=DEVICE)
clear_output() # Hide the loading messages
print("All models loaded and ready.")


# --- NEW FUNCTION: Prepare Audio File ---
def prepare_audio_file(source_type, input_data, status_label, progress_bar):
    """Prepares 'video_audio.mp3' from either a URL or an uploaded file."""
    # Clean up any old files first
    if os.path.exists(AUDIO_FILE): os.remove(AUDIO_FILE)
    if os.path.exists(TEMP_VIDEO_FILE): os.remove(TEMP_VIDEO_FILE)

    if source_type == 'URL':
        # Step 1: Downloading from URL (0% -> 33%)
        status_label.value = "Step 1/3: Downloading audio..."
        progress_bar.value = 0
        process = None
        yt_dlp_output = ""
        try:
            video_url = str(input_data)
            command = f"yt-dlp --progress --verbose -x --audio-format mp3 -o \"{AUDIO_FILE}\" \"{video_url}\""
            process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            progress_regex = re.compile(r"\[download\]\s+([0-9.]+)%")
            for line in iter(process.stdout.readline, ''):
                yt_dlp_output += line
                match = progress_regex.search(line)
                if match: percentage = float(match.group(1)); progress_bar.value = int(percentage * 0.33)
            process.stdout.close(); return_code = process.wait()
            if return_code != 0 or not os.path.exists(AUDIO_FILE):
                 error_lines = [line for line in yt_dlp_output.splitlines() if line.strip().startswith("ERROR:")]
                 specific_error = error_lines[-1].replace("ERROR:", "").strip() if error_lines else "Unknown download error."
                 if "Private video" in specific_error or "when logged-in" in specific_error: raise Exception("This video is private or requires login.")
                 elif "Video unavailable" in specific_error: raise Exception("Video is unavailable (may be deleted or region-locked).")
                 elif "Sign in" in specific_error or "--cookies" in specific_error: raise Exception("YouTube requires sign-in for this video (cookies needed). Please try a different URL.")
                 else: raise FileNotFoundError(f"Audio file not created. Reason: {specific_error}")
        except Exception as e:
            status_label.value = f"Error: Failed to download audio. {e}"; progress_bar.bar_style = 'danger'
            if process and process.poll() is None: process.terminate()
            return False
        progress_bar.value = 33
        return True

    elif source_type == 'Upload':
        # Step 1: Processing Uploaded File (0% -> 33%)
        status_label.value = "Step 1/3: Processing uploaded file..."
        progress_bar.value = 10
        try:
            if not input_data:
                raise ValueError("No file uploaded.")

            # input_data is a dict. Get the first (and only) file's content
            # The .value of a FileUpload widget is a tuple, get the first item
            file_info = input_data[0] # Get the first file's info dict
            content = file_info['content']

            # Save the uploaded video (e.g., .mp4, .mkv, .mov) temporarily
            with open(TEMP_VIDEO_FILE, 'wb') as f:
                f.write(content)

            # --- PROGRESS BAR FIX ---
            # Set a more informative label before the long, silent ffmpeg step
            status_label.value = "Step 1/3: Extracting audio (this may take a while)..."
            progress_bar.value = 20

            # Use ffmpeg to convert the temp video file to our audio file
            command = f"ffmpeg -i \"{TEMP_VIDEO_FILE}\" -vn -q:a 0 \"{AUDIO_FILE}\" -y"
            # Use subprocess.run, but capture output to hide it from the user
            subprocess.run(command, shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

            if not os.path.exists(AUDIO_FILE):
                raise FileNotFoundError("Audio could not be extracted from the uploaded file.")

        except Exception as e:
            status_label.value = f"Error processing upload: {e}"; progress_bar.bar_style = 'danger'
            return False
        finally:
            if os.path.exists(TEMP_VIDEO_FILE): os.remove(TEMP_VIDEO_FILE)

        progress_bar.value = 33
        return True

# --- HIGH-ACCURACY & STABLE Transcription ---
def transcribe_local_audio(status_label, progress_bar):
    """Transcribes 'video_audio.mp3' using the stable 'generate' method."""
    # Step 2: Transcribing (33% -> 66%)
    status_label.value = "Step 2/3: Preparing for transcription..."
    progress_bar.value = 40

    transcript_result = None
    try:
        status_label.value = "Step 2/3: Transcribing audio (High-Accuracy Mode)..."
        progress_bar.value = 45

        # --- Using the recommended 'generate' method ---
        # This is more stable and accurate, as per the warning.
        transcript_result = SPEECH_PIPE(
            AUDIO_FILE,
            return_timestamps=True,
            generate_kwargs={"task": "transcribe"}
        )
    except Exception as e:
        error_str = str(e).lower()
        if "cuda out of memory" in error_str or "out of memory" in error_str or "terminated" in error_str or "killed" in error_str:
             status_label.value = "Error: Ran out of memory during transcription. Your high-RAM environment may still be insufficient for this video."
        else: status_label.value = f"Error: Transcription failed unexpectedly. {e}"
        progress_bar.bar_style = 'danger'; return None, None
    finally:
        if os.path.exists(AUDIO_FILE): os.remove(AUDIO_FILE)
        # No model unloading needed, as it's pre-loaded
        gc.collect()
        if DEVICE == "cuda:0": torch.cuda.empty_cache()

    dialogues, timestamps = [], []
    if transcript_result and 'chunks' in transcript_result:
        for chunk in transcript_result['chunks']:
            dialogues.append(chunk['text'].strip())
            timestamps.append(chunk['timestamp'][0])
    progress_bar.value = 66

    return dialogues, timestamps

def build_and_search(query, dialogues, timestamps, status_label, progress_bar):
    """Performs search using the pre-loaded SBERT model."""
    if not dialogues:
        progress_bar.bar_style = 'danger'; status_label.value = "Error: No dialogue to search."; return pd.DataFrame()

    # --- Stage 1: Exact Search ---
    status_label.value = "Step 3/4: Searching for exact matches..."
    progress_bar.value = 75
    # --- Using Normalized Search for High Accuracy ---
    cleaned_query = re.sub(r'[^\w\s]', '', query).lower()
    exact_matches = []
    for i, dialogue in enumerate(dialogues):
        cleaned_dialogue = re.sub(r'[^\w\s]', '', dialogue).lower()
        if cleaned_query in cleaned_dialogue:
            exact_matches.append({"Timestamp": timestamps[i], "Dialogue Match": dialogue, "Match Type": "Exact Match"})

    # --- Stage 2: Semantic Search ---
    status_label.value = "Step 4/4: Encoding dialogue for semantic search..."
    progress_bar.value = 85
    embeddings = []
    batch_size_sbert = 128
    try:
        # SBERT_MODEL is already loaded
        for i in range(0, len(dialogues), batch_size_sbert):
            batch = dialogues[i:i+batch_size_sbert]
            batch_embeddings = SBERT_MODEL.encode(batch, convert_to_tensor=False)
            embeddings.append(batch_embeddings)

        if not embeddings:
            progress_bar.bar_style = 'warning'; status_label.value = "Warning: Could not encode dialogues."; return pd.DataFrame(exact_matches)

        embeddings = np.vstack(embeddings).astype('float32')

        status_label.value = "Step 4/4: Building & Performing semantic search..."
        progress_bar.value = 95
        dimension = embeddings.shape[1]
        faiss_index = faiss.IndexFlatL2(dimension)
        faiss_index.add(embeddings)
        query_embedding = SBERT_MODEL.encode([query], convert_to_tensor=False).astype('float32')
        distances, indices = faiss_index.search(query_embedding, k=10)

    except Exception as e:
        error_str = str(e).lower()
        if "cuda out of memory" in error_str or "out of memory" in error_str: status_label.value = "Error: Ran out of memory during search step."
        else: status_label.value = f"Error during semantic search: {e}"
        progress_bar.bar_style = 'danger'; return pd.DataFrame(exact_matches)

    suggestions = []
    found_dialogues = {em['Dialogue Match'] for em in exact_matches}
    if indices.size > 0:
        for i, idx in enumerate(indices[0]):
            if 0 <= idx < len(dialogues) and dialogues[idx] not in found_dialogues:
                suggestions.append({"Timestamp": timestamps[idx], "Dialogue Match": dialogues[idx], "Match Type": "Related Suggestion"})
                found_dialogues.add(dialogues[idx])
                if len(suggestions) >= 5: break

    final_results = exact_matches + suggestions
    df = pd.DataFrame(final_results)
    progress_bar.value = 100; progress_bar.bar_style = 'success'

    return df

def generate_timestamp_link(video_url, timestamp_sec):
    """Generates a platform-specific timestamp link if supported."""
    timestamp = int(timestamp_sec)
    if 'youtube.com' in video_url or 'youtu.be' in video_url:
        base_url = video_url.split('&t=')[0].split('#')[0]
        return f"{base_url.split('?')[0]}?t={timestamp}s" if '?' in base_url else f"{base_url}?t={timestamp}s"
    elif 'vimeo.com' in video_url:
        return f"{video_url.split('#')[0]}#t={timestamp}s"
    else: return video_url

# --- Advanced GUI for Colab ---

header = widgets.HTML(value="""<div style="text-align: center; margin-bottom: 20px;"><h1 style="font-size: 3em; font-weight: bold; color: #FFFFFF;">SceneSeek</h1><p style="font-size: 1.1em; color: #A0AEC0;">Find any scene in any online video by searching its dialogue.</p></div>""")

input_type_toggle = widgets.ToggleButtons(
    options=['Search by URL', 'Upload Video File'],
    description='',
    button_style='',
    tooltips=['Search a video from a web link', 'Search a video from your computer'],
)

url_input = widgets.Text(value='https://youtu.be/NHBMFCGzn-he?si=RKGWc0iPigALna_Ax', placeholder='Enter any Video URL...', layout=widgets.Layout(width='98%', height='40px'))

# --- BUG FIX: Define file_uploader in the global scope ---
file_uploader = widgets.FileUpload(
    accept='video/*',  # Accept all video formats
    multiple=False, # Changed to False, as we only process one file
    description='Select Video',
    layout=widgets.Layout(width='98%')
)

input_container = widgets.VBox([url_input])

query_input = widgets.Text(value="and the bottom line is you have", placeholder='What are you looking for?...', layout=widgets.Layout(width='98%', height='40px'))
run_button = widgets.Button(description='Find Scene', button_style='success', icon='search', layout=widgets.Layout(width='98%', height='45px', margin='10px 0 0 0'))
status_label = widgets.Label(value="", layout=widgets.Layout(width='98%'))
progress_bar = widgets.IntProgress(value=0, min=0, max=100, bar_style='info', style={'bar_color': '#60a5fa'}, layout=widgets.Layout(width='98%', display='none'))
output_area = widgets.Output(layout=widgets.Layout(width='98%'))

def on_toggle_change(change):
    """Hides/shows the correct input widget based on the toggle button."""
    # --- BUG FIX: Refer to the global file_uploader, don't create a new one ---
    if change['new'] == 'Search by URL':
        input_container.children = [url_input]
    else:
        # Clear the uploader's value when switching to it
        file_uploader.value.clear()
        input_container.children = [file_uploader]

input_type_toggle.observe(on_toggle_change, names='value')

def run_full_process(b):
    with output_area:
        clear_output(wait=True); run_button.disabled = True
        progress_bar.layout.display = 'flex'; progress_bar.value = 0; progress_bar.bar_style = 'info'
        status_label.value = "Starting..."

        source_type = 'URL' if input_type_toggle.value == 'Search by URL' else 'Upload'
        query = query_input.value
        input_data = None
        video_url_for_links = ""

        if source_type == 'URL':
            input_data = url_input.value
            video_url_for_links = input_data
            if not input_data or not query:
                status_label.value = "Error: Please provide both a Video URL and a search query."; progress_bar.bar_style = 'danger'; run_button.disabled = False; return
        else: # Upload
            # --- BUG FIX: Read from the global file_uploader ---
            input_data = file_uploader.value
            if not input_data or not query:
                status_label.value = "Error: Please upload a file and provide a search query."; progress_bar.bar_style = 'danger'; run_button.disabled = False; return

        if not prepare_audio_file(source_type, input_data, status_label, progress_bar):
            # --- UI RESET FIX (for failed upload) ---
            if source_type == 'Upload':
                file_uploader.value.clear() # Clear the uploader on failure
            run_button.disabled = False; return

        dialogues, timestamps = transcribe_local_audio(status_label, progress_bar)

        if dialogues is not None:
            results_df = build_and_search(query, dialogues, timestamps, status_label, progress_bar)
            if progress_bar.bar_style != 'danger': status_label.value = "Search complete."

            if not results_df.empty:

                if source_type == 'URL':
                    results_df['Jump Link'] = results_df.apply(lambda r: generate_timestamp_link(video_url_for_links, r['Timestamp']), axis=1)
                    display_df = results_df[['Timestamp', 'Dialogue Match', 'Match Type', 'Jump Link']]
                    formatters = {
                        'Timestamp': lambda x: time.strftime('%M:%S', time.gmtime(x)),
                        'Jump Link': lambda x: f'<a href="{x}" target="_blank" style="color:#60a5fa;font-weight:bold;">Jump to Scene</a>',
                        'Match Type': lambda mt: f'<b style="color: #48BB78;">{mt}</b>' if mt == 'Exact Match' else f'<span style="color: #A0AEC0;">{mt}</span>'
                    }
                else:
                    display_df = results_df[['Timestamp', 'Dialogue Match', 'Match Type']]
                    formatters = {
                        'Timestamp': lambda x: time.strftime('%M:%S', time.gmtime(x)),
                        'Match Type': lambda mt: f'<b style="color: #48BB78;">{mt}</b>' if mt == 'Exact Match' else f'<span style="color: #A0AEC0;">{mt}</span>'
                    }

                html = display_df.to_html(escape=False, index=False, formatters=formatters)
                display(HTML(f"""<style>.dataframe{{width:100%;border-collapse:collapse;color:#E5E7EB}} .dataframe th{{background-color:#374151;padding:12px;text-align:left;border-bottom:1px solid #4B5563}} .dataframe td{{padding:12px;border-bottom:1px solid #4B5563}} .dataframe tr:hover{{background-color:#4B5563}}</style><h2 style="color:#FFFFFF;font-size:1.5em;margin-top:20px;">Search Results</h2>{html}"""))
            elif progress_bar.bar_style != 'danger': status_label.value = "No relevant results found for your query."

        # --- UI RESET FIXES ---
        # 1. Clear the file uploader value
        if source_type == 'Upload':
            if file_uploader.value:
                file_uploader.value.clear() # Clear the widget's value

        # 2. Do NOT reset the toggle button, as this caused the race condition.
        # Let the user decide what to do next.

        run_button.disabled = False

run_button.on_click(run_full_process)
gui = widgets.VBox([
    header,
    input_type_toggle,
    input_container,
    widgets.Label("What are you looking for?:"),
    query_input,
    run_button,
    status_label,
    progress_bar,
    output_area
], layout=widgets.Layout(display='flex', flex_flow='column', align_items='center', width='100%'))

if DEVICE == "cpu": display(HTML("""<div style="background-color:#4A5568;border-left:5px solid #F56565;padding:15px;margin-bottom:20px;color:#FFFFFF;"><h3 style="margin-top:0;"><b>Performance Warning: No GPU Detected</b></h3><p>For a massive speed improvement, please go to <b>Runtime</b> &rarr; <b>Change runtime type</b> and select a <b>GPU</b> accelerator (like T4), then re-run this cell.</p></div>"""))
display(gui)



All models loaded and ready.


In [ ]:
!pip install flask flask-cors pyngrok


In [ ]:
  from pyngrok import ngrok
  ngrok.set_auth_token("33TIrwqFrV5bFkOXn2WlCCest3s_5zjyMd63T9ykGR2shx7Yu")


In [ ]:
%%bash
mkdir -p /content/scene_seek_app


In [ ]:
%%bash
cat >/content/scene_seek_app/index.html <<'EOF'
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>SceneSeek</title>
<style>
  body {
    margin:0; padding:0;
    height:100vh; display:flex; align-items:center; justify-content:center;
    background:
      radial-gradient(1000px 600px at 20% -10%, rgba(255,255,255,0.05), transparent 60%),
      #0a0a0a;
    color:#fff; font-family: Inter, sans-serif;
  }
  .glass {
    width:90vw; max-width:1100px;
    padding:32px; border-radius:32px;
    background:rgba(255,255,255,0.03);
    border:1px solid rgba(255,255,255,0.1);
    backdrop-filter: blur(22px);
    box-shadow:0 20px 80px rgba(0,0,0,0.6);
  }
  h1 { font-size:44px; margin-bottom:10px; text-align:center; }
  p { text-align:center; opacity:0.8; }

  .section { margin:16px 0; }
  label { font-weight:600; display:block; margin-bottom:8px; }

  input, button {
    width:100%; padding:16px; border-radius:16px;
    background:#0f0f0f; color:#fff;
    border:1px solid rgba(255,255,255,0.2);
  }
  button {
    background:#fff; color:#000; font-weight:800; border:none; cursor:pointer;
  }
  #bar {
    height:8px; width:0%; background:#fff;
    border-radius:999px; margin-top:10px; transition:width .2s;
  }

  table { width:100%; margin-top:20px; border-collapse:collapse; }
  th,td { padding:12px; border-bottom:1px solid rgba(255,255,255,0.1); }
  th { color:#ccc; }
</style>
</head>

<body>
<div class="glass">
  <h1>SceneSeek</h1>
  <p></p>

  <div class="section">
    <label>Video URL</label>
    <input id="url" placeholder="https://youtu.be/...">
  </div>

  <div class="section">
    <label>Query</label>
    <input id="query" placeholder="Dialogue to search">
  </div>

  <button id="runBtn">Find Scene</button>

  <div id="bar"></div>

  <table id="results"></table>
</div>

<script>
runBtn.onclick = async () => {
  const url = document.getElementById("url").value;
  const query = document.getElementById("query").value;
  if(!url || !query){ alert("Enter URL + query"); return; }

  document.getElementById("bar").style.width = "30%";

  const res = await fetch("/api/run-url", {
    method:"POST",
    headers:{ "Content-Type":"application/json" },
    body:JSON.stringify({ url, query })
  });

  const data = await res.json();
  document.getElementById("bar").style.width = "100%";

  const table = document.getElementById("results");
  let html = `
  <tr><th>Time</th><th>Line</th><th>Type</th><th>Jump</th></tr>
  `;
  for(const r of data.rows){
    html += `
    <tr>
      <td>${r.Timestamp_mmss}</td>
      <td>${r["Dialogue Match"]}</td>
      <td>${r["Match Type"]}</td>
      <td>${r["Jump Link"] ? `<a href="${r["Jump Link"]}" target="_blank">Open</a>` : "-"}</td>
    </tr>`;
  }
  table.innerHTML = html;
};
</script>
</body>
</html>
EOF


In [ ]:
from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS
from pyngrok import ngrok
import time

app = Flask(__name__, static_folder="/content/scene_seek_app")
CORS(app)

class FakeLabel:
    def __init__(self): self.value=""
class FakeBar:
    def __init__(self): self.value=0; self.bar_style="info"

@app.route("/")
def index():
    return send_from_directory(app.static_folder, "index.html")


@app.route("/api/run-url", methods=["POST"])
def run_url_html():
    body = request.json
    video_url = body["url"]
    query = body["query"]

    status = FakeLabel()
    progress = FakeBar()

    ok = prepare_audio_file("URL", video_url, status, progress)
    if not ok:
        return jsonify(ok=False, error=status.value)

    dialogues, timestamps = transcribe_local_audio(status, progress)
    if dialogues is None:
        return jsonify(ok=False, error=status.value)

    df = build_and_search(query, dialogues, timestamps, status, progress)

    rows=[]
    for _, r in df.iterrows():
        ts = int(r["Timestamp"])
        rows.append({
            "Timestamp_mmss": time.strftime('%M:%S', time.gmtime(ts)),
            "Dialogue Match": r["Dialogue Match"],
            "Match Type": r["Match Type"],
            "Jump Link": generate_timestamp_link(video_url, ts)
        })

    return jsonify(ok=True, rows=rows)

# Start server via ngrok
public_url = ngrok.connect(5000)
print("✅ PUBLIC URL:", public_url)

app.run(port=5000)


✅ PUBLIC URL: NgrokTunnel: "https://sublanate-unrentable-elenora.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [20/Jan/2026 18:01:02] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [20/Jan/2026 18:01:03] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [20/Jan/2026 18:06:46] "POST /api/run-url HTTP/1.1" 200 -
